# E2.7 · Documentation that survives supervision

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.6 · Incident and disclosure obligations](https://spbreed.github.io/cyber-commons/lessons/E2.6.html)**.

| | |
|---|---|
| Open-source tooling | OSCAL, Model Cards |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Documentation written for an auditor and documentation written for engineers are usually two documents that disagree. The one that survives supervision is the one generated from the same source as the system.

> **At CyberTravels.** The document describing TripBot's oversight has to survive a supervisor asking when that oversight last operated. R2.

## 2 · The framework

```
   two documents that disagree
   +------------------+     +---------------------+
   | for the auditor  |     | for the engineers   |
   +------------------+     +---------------------+

   one document, generated from the system's own source
   +----------------------------------------------+
   | model card . decision record . control        |
   | narrative — all from the register and the CI  |
   +----------------------------------------------+
```

Documentation survives supervision when it points at machine-generated evidence
rather than restating intent.

The difference is not length or formality. It is whether each sentence names
three things:

- a **control** that operates,
- an **artefact** it produces,
- a **date** on which that artefact was last produced.

A sentence with all three can be checked. A sentence with none of them is a
statement of intent, and a supervisor's next question makes that visible
immediately.

Intent statements are not forbidden — some things genuinely are aspirations. The
failure is presenting them as controls. Label them, and the rest of the document
becomes more credible rather than less.

## 3 · Demo — the same policy paragraph, two ways

In [ ]:
import re, time
now = time.time(); DAY = 86400

WEAK = """
Our AI systems are subject to appropriate oversight and controls. Access is
granted on a least-privilege basis and reviewed periodically. Agents are
monitored for anomalous behaviour and we maintain comprehensive logging.
"""

STRONG = """
Agent identities are distinct from human identities (AC-1). Evidence: gateway
logs containing an act chain for every action; sampled monthly, last test
2026-08-13, valid 30d.

Delegated authority narrows at every hop (AC-2). Evidence: the token exchange
refuses widening; regression cases IDN-01/IDN-04 run on every release, last run
2026-08-15.

Autonomy above L2 requires approval for privileged tools (SB-2). Evidence: tool
policy in git; 90-day denial log attached, last reviewed 2026-07-06.
"""

CONTROL_RE = re.compile(r"\b([A-Z]{2}-\d)\b")
DATE_RE    = re.compile(r"\b20\d{2}-\d{2}-\d{2}\b")
ARTEFACT_RE = re.compile(r"\b(log|logs|sample|report|test|cases|policy|record)\b", re.I)

def score_paragraph(text):
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    rows = []
    for s in sentences:
        rows.append({"has_control": bool(CONTROL_RE.search(s)),
                     "has_artefact": bool(ARTEFACT_RE.search(s)),
                     "has_date": bool(DATE_RE.search(s)),
                     "text": s[:56]})
    checkable = [r for r in rows if r["has_control"] and r["has_artefact"]]
    return rows, len(checkable), len(rows)

for label, text in (("WEAK", WEAK), ("STRONG", STRONG)):
    rows, checkable, total = score_paragraph(text)
    print(f"=== {label} — {checkable}/{total} sentences checkable ===")
    for r in rows:
        marks = ("C" if r["has_control"] else "-") + \
                ("A" if r["has_artefact"] else "-") + \
                ("D" if r["has_date"] else "-")
        print(f"   [{marks}] {r['text']}")
    print()
print("C = names a control · A = names an artefact · D = carries a date")

## 4 · Where it breaks — the follow-up question

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the follow-up question</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">answerable from the document?</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">“appropriate oversight” — show me the last time it operated.</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">“reviewed periodically” — what period, and when was the last one?</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">“comprehensive logging” — produce one action&#x27;s full record.</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">“act chain for every action” — produce the August sample.</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">yes</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Three of four cannot be answered, and by the time the fourth is asked the person asking doubts that one too. Vague language does not fail on its own line; it fails the lines around it.</div>

## 5 · The control — verify the document against live control state

In [ ]:
from dataclasses import dataclass
@dataclass
class ControlTest:
    cid: str; passed: bool; tested_at: float; valid_for_days: float
    def state(self, at):
        if (at - self.tested_at)/DAY > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

TESTS = {t.cid: t for t in [
 ControlTest("AC-1", True, now -  3*DAY, 30),
 ControlTest("AC-2", True, now -  1*DAY, 30),
 ControlTest("SB-2", True, now - 40*DAY, 30)]}

cited = CONTROL_RE.findall(STRONG)
print(f"controls cited in the document: {sorted(set(cited))}\n")
print(f"{'control':9s}{'state now':12s}{'document claim still true?':>28}")
print("-" * 52)
stale = []
for cid in sorted(set(cited)):
    st = TESTS[cid].state(now) if cid in TESTS else "NO EVIDENCE"
    ok = st == "PASS"
    if not ok: stale.append(cid)
    print(f"{cid:9s}{st:12s}{str(ok):>28}")
print(f"\n{len(stale)} cited control(s) no longer evidenced: {stale}")
print("A document that cites controls can be CHECKED against live state.")
print("A document of intent cannot go stale, because it never said anything.")
assert stale

## What you just proved

The weak paragraph has zero checkable sentences; the strong one has three, each naming a control, an artefact and a date. Three of four supervisor follow-ups are unanswerable from the weak version. Verifying the strong document against live control state finds SB-2 stale, so one of its claims is no longer true — which is only detectable because the document named a control.

## Your turn

Rewrite one paragraph of your AI policy in the strong shape. Any sentence that cannot name an artefact is intent — label it as such rather than deleting it, and the document gets more credible.

---

**Next → [E2.8 · Auditability of autonomous action](https://spbreed.github.io/cyber-commons/lessons/E2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*